In [1]:
!pip install -q python-dotenv

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Import necessary libraries
import os
from dotenv import load_dotenv
from transformers import pipeline
import librosa
import datetime

# Load environment variables from .env file
load_dotenv("/content/drive/MyDrive/AI Datasets/key/.env")

# Load the API key from the environment
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

# Load the ASR model directly from Hugging Face
asr = pipeline(task="automatic-speech-recognition",
               model="openai/whisper-large",
               generate_kwargs = {"language":"en","task": "transcribe"},
               return_timestamps=True)

# Ensure the sampling rate is correct
sampling_rate = asr.feature_extractor.sampling_rate


config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.85k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
def split_audio(audio, sr, chunk_duration):
    """
    Split audio into smaller chunks.

    Args:
        audio (numpy.ndarray): The audio array.
        sr (int): The sampling rate of the audio.
        chunk_duration (int): Duration of each chunk in seconds.

    Returns:
        list: List of audio chunks (numpy arrays).
    """
    total_duration = librosa.get_duration(y=audio, sr=sr)
    chunks = []
    for i in range(0, int(total_duration), chunk_duration):
        start_sample = i * sr
        end_sample = min((i + chunk_duration) * sr, len(audio))
        chunks.append(audio[start_sample:end_sample])
    return chunks

In [5]:
def process_audio_files():
    audio_dir = '/content/drive/MyDrive/AI Datasets/Supertails/data/audio'
    transcription_dir = '/content/drive/MyDrive/AI Datasets/Supertails/data/transcription'

    if not os.path.exists(transcription_dir):
        os.makedirs(transcription_dir)

    processed_files = set()

    # List all audio files in the directory
    audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.mp3', '.wav'))]

    for audio_file in audio_files:
        if audio_file in processed_files:
            continue  # Skip already processed files

        filepath = os.path.join(audio_dir, audio_file)

        # Load the audio file
        audio, sr = librosa.load(filepath, sr=sampling_rate)

        # Split the audio into smaller chunks
        audio_chunks = split_audio(audio, sr, chunk_duration=300)

        # Transcribe each chunk and combine the results
        transcription = ""
        for i, chunk in enumerate(audio_chunks):
            print(f"Transcribing chunk {i+1}/{len(audio_chunks)} for file {audio_file}...")
            # Ensure the chunk is a NumPy array of type float32
            chunk = chunk.astype('float32')

            # Pass the chunk directly to the ASR pipeline
            output = asr(chunk)
            transcription += output["text"] + " "

        # Store the transcribed text in a file
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        transcription_filename = f'transcription_{timestamp}_{audio_file}.txt'
        transcription_path = os.path.join(transcription_dir, transcription_filename)

        with open(transcription_path, 'w') as f:
            f.write(transcription)  # Write the raw transcription directly

        # Mark the file as processed
        processed_files.add(audio_file)
        print(f"Processed and transcribed: {audio_file}")


In [6]:
# Run the audio processing function
if __name__ == "__main__":
    process_audio_files()

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


Transcribing chunk 1/1 for file ashwini_emmanuel_supertails_in__Pharmacy_OB__320__-1__9483505997__2025-01-12_10-15-23.mp3...
Processed and transcribed: ashwini_emmanuel_supertails_in__Pharmacy_OB__320__-1__9483505997__2025-01-12_10-15-23.mp3
Transcribing chunk 1/1 for file mehak_nazar_supertails_in__Pharmacy_OB__320__-1__7838433345__2025-01-09_18-56-21.mp3...


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Processed and transcribed: mehak_nazar_supertails_in__Pharmacy_OB__320__-1__7838433345__2025-01-09_18-56-21.mp3
Transcribing chunk 1/1 for file mounika_latha_supertails_in__Pharmacy_OB__320__-1__8073884941__2025-01-10_17-01-42.mp3...
Processed and transcribed: mounika_latha_supertails_in__Pharmacy_OB__320__-1__8073884941__2025-01-10_17-01-42.mp3
Transcribing chunk 1/1 for file mounika_latha_supertails_in__Pharmacy_OB__320__-1__8013844702__2025-01-12_11-50-59.mp3...
Processed and transcribed: mounika_latha_supertails_in__Pharmacy_OB__320__-1__8013844702__2025-01-12_11-50-59.mp3
Transcribing chunk 1/2 for file naheeda_begum_supertails_in__Pharmacy_OB__320__-1__7428015866__2025-01-10_18-47-28 (1).mp3...


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Transcribing chunk 2/2 for file naheeda_begum_supertails_in__Pharmacy_OB__320__-1__7428015866__2025-01-10_18-47-28 (1).mp3...


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Processed and transcribed: naheeda_begum_supertails_in__Pharmacy_OB__320__-1__7428015866__2025-01-10_18-47-28 (1).mp3
